In [ ]:
# Prefer the advanced app: python run_advanced.py
# This notebook is a minimal classic live demo.

import cv2
import numpy as np
import tkinter as tk
from PIL import Image, ImageTk
from keras.models import load_model
from keras.layers import DepthwiseConv2D
from cvzone.HandTrackingModule import HandDetector


class CustomDepthwiseConv2D(DepthwiseConv2D):
    def __init__(self, **kwargs):
        kwargs.pop("groups", None)
        super().__init__(**kwargs)


classifier = load_model(
    "best_model.keras",
    custom_objects={"DepthwiseConv2D": CustomDepthwiseConv2D},
)
cap = cv2.VideoCapture(0)
detector = HandDetector(maxHands=1)

offset = 20
img_size = 224
labels = ["GOODBYE", "HELLO", "I LOVE YOU", "THANK YOU"]
threshold = 0.75
stable_frames_required = 5
stable_count = 0
last_label = None

root = tk.Tk()
root.title("Sign Language Detection")
root.geometry("900x500")

left_frame = tk.Frame(root, width=300, height=500)
left_frame.pack(side="left", fill="y")
text_display = tk.Text(
    left_frame, font=("Helvetica", 16), wrap="word", spacing3=10, width=30, height=25
)
text_display.pack(pady=20)

right_frame = tk.Frame(root, width=600, height=500)
right_frame.pack(side="right", fill="both", expand=True)
camera_label = tk.Label(right_frame)
camera_label.pack()


def update_frame():
    global stable_count, last_label

    success, img = cap.read()
    if not success:
        root.after(15, update_frame)
        return

    hands, display = detector.findHands(img)

    if hands:
        x, y, w, h = hands[0]["bbox"]
        y1, y2 = max(0, y - offset), min(img.shape[0], y + h + offset)
        x1, x2 = max(0, x - offset), min(img.shape[1], x + w + offset)
        crop = img[y1:y2, x1:x2]

        if crop.size:
            batch = np.expand_dims(
                cv2.resize(crop, (img_size, img_size)).astype(np.float32) / 255.0,
                axis=0,
            )
            prediction = classifier.predict(batch, verbose=0)[0]
            index = int(np.argmax(prediction))
            confidence = float(prediction[index])

            if confidence > threshold:
                detected = labels[index]
                stable_count = stable_count + 1 if detected == last_label else 1
                last_label = detected
                if stable_count >= stable_frames_required:
                    text_display.insert(tk.END, f"{detected}\n")
                    text_display.see("end")
                    stable_count = 0
            else:
                stable_count = 0
                last_label = None

    img_tk = ImageTk.PhotoImage(
        Image.fromarray(cv2.cvtColor(display, cv2.COLOR_BGR2RGB))
    )
    camera_label.imgtk = img_tk
    camera_label.configure(image=img_tk)
    root.after(15, update_frame)


def on_close():
    cap.release()
    root.destroy()


root.protocol("WM_DELETE_WINDOW", on_close)
update_frame()
root.mainloop()
